# Wind disturbance sweep

Runs the figure-8 in SITL across a grid of Gazebo wind conditions and reports how tracking
degrades. Produces `media/disturbance_recovery.gif` (companion) and the README's
tracking-error-vs-wind plot.

Sweep grid (§12.3):
- wind speed: 0, 2, 4, 6, 8 m/s
- wind direction: 0, 45, 90 deg relative to the long axis of the 8
- gust: off / on (Gazebo `wind` plugin with a Gaussian gust component)
- 3 repeats per cell, different RNG seeds

This notebook orchestrates SITL runs — it is slow (hours) and is NOT part of CI. Cache every
run to `analysis/output/sweep/` so re-plotting never re-flies.


## 1. Setup and sweep definition

Defines the grid, the deterministic cache key, and the helpers the later cells share. Run
this cell first; everything below depends on it.


In [ ]:
import hashlib
import math
import os
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:  # tqdm is not in requirements.txt; degrade gracefully
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(seq, **kwargs):
        return seq

# Make analysis/ importable regardless of the kernel's cwd (bag_utils owns the bag I/O).
if Path.cwd().name == "analysis":
    sys.path.insert(0, str(Path.cwd()))
else:
    sys.path.insert(0, str(Path.cwd() / "analysis"))
from bag_utils import read_topics

SWEEP_DIR = Path("output/sweep")
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
WORLDS_DIR = SWEEP_DIR / "worlds"
RESULTS_CSV = SWEEP_DIR / "results.csv"

# The explicit grid (§12.3): 5 speeds x 3 directions x 2 gust states x 3 seeds = 90 trials.
GRID = [
    {"wind": w, "direction": d, "gust": g, "seed": s}
    for w in (0, 2, 4, 6, 8)
    for d in (0, 45, 90)
    for g in ("off", "on")
    for s in (1, 2, 3)
]

TRIAL_TIMEOUT_S = 12 + 2 * 8 + 40      # takeoff_wait + laps*period + margin
LAPS = 2
TAKEOFF_ALTITUDE = 1.5
TUBE_RADIUS_M = 0.5                    # "in the tube" = pn <= 0.5 m (failure boundary)

# Deterministic run id from the parameters — the resume key and the cache file stem.
def run_id(p: dict) -> str:
    key = f"{p['wind']}_{p['direction']}_{p['gust']}_seed{p['seed']}"
    return hashlib.sha1(key.encode()).hexdigest()[:12]


## 2. Run one SITL trial

Smoke-tests `run_trial` on a single cell before launching the full 90-run sweep. Expect it
to take roughly 60 s (12 s takeoff allowance + 2 × 8 s laps + margin) and to print a row
with `success=True`. Set `SINGLE = False` once you trust it.


In [ ]:
def write_world(p: dict, id_: str):
    """Copy the base Gazebo world and inject the wind plugin's gust parameters.

    Returns the world path, or None for non-gust trials (steady wind alone is applied via
    PX4_GZ_WIND). UNVERIFIED: the wind-plugin element names and the world-frame axis
    convention must be confirmed against the pinned PX4/Gazebo versions the first time this
    runs (§2.1 risk register) — every cell uses the same convention, so the sweep's
    relative comparisons hold regardless.
    """
    if not p["gust"]:
        return None
    base = os.environ.get("PX4_GZ_WORLD", "default")
    if Path(base).suffix == ".sdf" and Path(base).is_file():
        world = Path(base)
    else:
        px4_root = os.environ.get("PX4_ROOT")
        candidates = ([Path(px4_root)] if px4_root else []) + [
            Path.home() / "PX4-Autopilot", Path.home() / "src" / "PX4-Autopilot",
        ]
        found = next((r / "Tools/simulation/gz/worlds" / f"{base}.sdf"
                      for r in candidates if (r / "Tools/simulation/gz/worlds"
                                             / f"{base}.sdf").is_file()), None)
        if found is None:
            raise RuntimeError(
                f"cannot locate Gazebo world '{base}' for the gust trial — set PX4_ROOT "
                f"to the PX4-Autopilot source tree or PX4_GZ_WORLD to a world .sdf path")
        world = found

    WORLDS_DIR.mkdir(parents=True, exist_ok=True)
    out = WORLDS_DIR / f"wind_{id_}.sdf"
    text = world.read_text()
    # Steady wind inside the world (as well as via PX4_GZ_WIND): the injected direction is
    # applied to the wind plugin's velocity vector. The gust block is appended inside the
    # plugin only when one is present; otherwise the trial runs with steady wind and the
    # result row notes it.
    gust_block = ("\n  <gust>\n"
                  "    <mean>1.0</mean>   <!-- multiplier on the steady wind -->\n"
                  "    <var>0.6</var>     <!-- Gaussian gust variance -->\n"
                  "    <period>5.0</period>\n"
                  "  </gust>\n")
    if "<gust" not in text:
        if "<plugin" not in text:
            text += "\n<!-- wind plugin: not present in the base world; gust inactive -->\n"
        else:
            text = text.replace("</plugin>", gust_block + "</plugin>", 1)
    out.write_text(text)
    return out


def newest_bag():
    """The analysis/output/figure8_* directory most recently written by the launch."""
    candidates = sorted(Path("output").glob("figure8_*")) + sorted(
        Path("../analysis/output").glob("figure8_*"))
    return max(candidates, key=lambda d: d.stat().st_mtime, default=None)


def score_bag(bag: Path) -> dict:
    """Per-trial metrics from the recorded bag (same scoring as scripts/assert_hover.py)."""
    data = read_topics(bag, ("/nmpc_node/status", "/fmu/out/vehicle_local_position"))
    status = data.get("/nmpc_node/status", [])
    if not status:
        return {"rms_error_m": np.nan, "max_error_m": np.nan, "solver_failures": np.nan,
                "tube_fraction": np.nan, "tracked_s": 0.0}
    tracking = [(t, m) for t, m in status if m.controller_state == 3]
    if not tracking:
        return {"rms_error_m": np.nan, "max_error_m": np.nan, "solver_failures": np.nan,
                "tube_fraction": np.nan, "tracked_s": 0.0}
    t0 = tracking[0][0]
    win = [(t, m) for t, m in status if t0 <= t <= t0 + 60 * 1e9]
    errs = [m.position_error_norm for _, m in win]
    return {
        "rms_error_m": float(np.sqrt(np.mean(np.square(errs)))) if errs else np.nan,
        "max_error_m": float(max(errs)) if errs else np.nan,
        "solver_failures": int(sum(1 for _, m in win if m.solver.status != 0)),
        "tube_fraction": float(np.mean(np.array(errs) <= TUBE_RADIUS_M)) if errs else np.nan,
        "tracked_s": (win[-1][0] - t0) / 1e9 if win else 0.0,
    }


def run_trial(p: dict) -> dict:
    """Run one SITL figure-8 with the wind injected; returns a result row dict.

    A crashed or timed-out trial is a recorded FAILURE row, not an exception — a sweep that
    dies on trial 47 of 90 is useless (§12.3). The process group is killed cleanly (SIGINT,
    then SIGKILL) so the bag is preserved for post-mortem.
    """
    id_ = run_id(p)
    world = write_world(p, id_)
    env = dict(os.environ)
    u = p["wind"] * math.cos(math.radians(p["direction"]))
    v = p["wind"] * math.sin(math.radians(p["direction"]))
    env["PX4_GZ_WIND"] = f"{u:.3f},{v:.3f},0"
    if world is not None:
        env["PX4_GZ_WORLD"] = str(world)

    cmd = ["ros2", "launch", "uav_mpc", "figure8.launch.py",
           "record:=true", "headless:=true", "rviz:=false", "aggressive:=false",
           f"laps:={LAPS}", "takeoff_wait:=12.0"]
    row = {"id": id_, **p, "success": True, "reason": "", "bag": "", "world": ""}
    try:
        proc = subprocess.Popen(cmd, env=env, stdout=subprocess.DEVNULL,
                                stderr=subprocess.DEVNULL, start_new_session=True)
    except FileNotFoundError as exc:
        raise RuntimeError(
            "'ros2' not on PATH — run this notebook from a sourced workspace with uav_mpc "
            "installed (colcon build && source install/setup.bash)") from exc
    try:
        proc.wait(timeout=TRIAL_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGINT)
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
        row.update(success=False, reason="timeout")
    bag = newest_bag()
    if bag is None:
        row.update(success=False, reason=(row["reason"] or "no bag produced"))
        for k in ("rms_error_m", "max_error_m", "solver_failures", "tube_fraction",
                  "tracked_s"):
            row[k] = np.nan
        return row
    row["bag"] = str(bag)
    row.update(score_bag(bag))
    if not row["success"]:
        row["reason"] = "timeout"
    return row


# --- smoke test -------------------------------------------------------------
SINGLE = True  # smoke test: a single no-wind trial, then stop
if SINGLE:
    trial = run_trial({"wind": 0, "direction": 0, "gust": "off", "seed": 1})
    print(pd.Series(trial).to_string())
    print(f"bag: {trial['bag']}")


## 3. Execute the sweep

Iterates the grid, skipping cached trials (resumable after an interrupt), and appends each
result row to `results.csv`. ~90 × 60 s ≈ 1.5 h on one machine. Set `SINGLE = False` first.


In [ ]:
results = []
for p in tqdm(GRID, desc="wind sweep"):
    id_ = run_id(p)
    cache = SWEEP_DIR / f"{id_}.json"
    if cache.is_file():  # resumable: never re-fly a completed cell
        row = pd.read_json(cache, typ="series").to_dict()
        results.append(row)
        continue
    row = run_trial(p)
    row["id"] = id_
    pd.Series(row).to_json(cache)
    results.append(row)

df = pd.DataFrame(results)
df.to_csv(RESULTS_CSV, index=False)
print(f"completed {len(df)} trials -> {RESULTS_CSV}")
print(df.groupby(["wind", "direction", "gust"])["success"].mean().unstack("gust"))


## 4. Aggregate

Per-cell mean ± std across the 3 repeats (report both — the spec wants the spread, not just
the mean), plus the failure bookkeeping: trials that crashed or timed out appear here as
`FAILURE` rows and are excluded from the error statistics but counted in the boundary plot.


In [ ]:
df = pd.read_csv(RESULTS_CSV)

ok = df[df["success"] & df["rms_error_m"].notna()]
agg = (ok.groupby(["wind", "direction", "gust"])
       .agg(rms_mean=("rms_error_m", "mean"), rms_std=("rms_error_m", "std"),
            max_mean=("max_error_m", "mean"), tube_mean=("tube_fraction", "mean"))
       .reset_index())
fail = df[~df["success"]]
print(f"total={len(df)}  ok={len(ok)}  failed={len(fail)}")
if len(fail):
    print(fail[["wind", "direction", "gust", "seed", "reason"]].to_string(index=False))
print(agg.sort_values(["wind", "direction", "gust"]).to_string(index=False))


## 5. Figures

(a) RMS position error vs wind speed, one line per direction, error bars = ±1 std across
repeats. (b) Heatmap of mean RMS over the wind × direction plane. (c) Failure boundary —
for each wind speed, the fraction of trials where the solver reports infeasibility or the
vehicle leaves the 0.5 m tube. Saved next to the notebook in `../media/` for the README.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) RMS vs wind speed, one line per direction, error bars across repeats.
for d in sorted(agg["direction"].unique()):
    sub = agg[agg["direction"] == d].sort_values("wind")
    axes[0].errorbar(sub["wind"], sub["rms_mean"], yerr=sub["rms_std"],
                     marker="o", capsize=3, label=f"{d:.0f} deg")
axes[0].set(xlabel="wind [m/s]", ylabel="RMS position error [m]",
            title="RMS error vs wind (figure-8)")
axes[0].axhline(TUBE_RADIUS_M, color="gray", ls="--", lw=1, label="0.5 m tube")
axes[0].legend()

# (b) Heatmap: wind x direction, mean RMS over the two gust states.
piv = agg.groupby(["wind", "direction"])["rms_mean"].mean().unstack()
im = axes[1].imshow(piv, aspect="auto", origin="lower", cmap="magma_r")
axes[1].set_xticks(range(len(piv.columns)), piv.columns)
axes[1].set_yticks(range(len(piv.index)), piv.index)
axes[1].set(xlabel="direction [deg]", ylabel="wind [m/s]",
            title="RMS error [m]")
fig.colorbar(im, ax=axes[1], label="RMS [m]")

# (c) Failure boundary: for each wind speed, the fraction of trials that crashed, timed
# out, or left the 0.5 m tube. Missing rms (failed) rows count as outside the tube.
failc = (df.assign(outside=lambda d: ~d["success"] | d["rms_error_m"].isna())
         .groupby("wind")["outside"].mean())
axes[2].plot(failc.index, failc.values, marker="o")
axes[2].fill_between(failc.index, 0, failc.values, alpha=0.2)
axes[2].set(xlabel="wind [m/s]", ylabel="failure fraction",
            title="failure boundary (crash / tube exit)", ylim=(0, 1.05))

fig.tight_layout()
out = Path("../media") / "tracking_error_vs_wind.png"
out.parent.mkdir(exist_ok=True)
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"saved {out}")


## 6. Recovery GIF frames

Replays the strongest gust trial (6 m/s, gust on, seed 1) as an animated GIF — ground track
with the position-error trace overlaid, plus a live error timeline. Requires the bag from a
completed gust trial in `output/sweep/` and Pillow for the GIF writer. Frames are rendered
from the bag, so this cell is deterministic given the cache.


In [ ]:
import matplotlib.animation as anim  # noqa: E402

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.set(xlabel="x [m]", ylabel="y [m]",
        title=f"ground track + position error — {trial['id']}")
ax1.set_aspect("equal")
ax2.set(xlabel="t [s]", ylabel="position error [m]",
        title="|e| with 0.5 m tube")
ax2.axhline(0.5, color="gray", ls="--", lw=1)

trace_l, = ax1.plot([], [], color="tab:blue", lw=1)
trace_r, = ax1.plot([], [], "o", color="tab:red", ms=6)
err_line, = ax2.plot([], [], color="tab:orange", lw=1)
frame_text = ax1.text(0.02, 0.95, "", transform=ax1.transAxes, va="top")

frames = np.arange(0.0, min(lt[-1], ts[-1]), 0.2)

def update(k):
    t = frames[int(k)]
    mask = lt <= t
    trace_l.set_data(lx[mask], ly[mask])
    if mask.any():
        trace_r.set_data([lx[mask][-1]], [ly[mask][-1]])
    m2 = ts <= t
    err_line.set_data(ts[m2], errs[m2])
    frame_text.set_text(f"t = {t:.1f} s")
    return trace_l, trace_r, err_line, frame_text

writer = anim.PillowWriter(fps=10)
out = Path("../media") / "disturbance_recovery.gif"
out.parent.mkdir(exist_ok=True)
ani = anim.FuncAnimation(fig, update, frames=len(frames), blit=True)
ani.save(out, writer=writer, dpi=100)
print(f"saved {out} ({len(frames)} frames)")
